# Abbiategrasso: popolazione e consumo di suolo

Primo notebook locale minimale.

Obiettivo: leggere una fotografia semplice di Abbiategrasso usando due output gia' presenti nel workspace:

- popolazione residente ISTAT
- consumo di suolo ISPRA


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd


def find_workspace_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'dataset-incubator').exists() and (candidate / 'abbiategrasso-data-notes').exists():
            return candidate
    raise FileNotFoundError('Workspace root non trovato')


workspace = find_workspace_root(Path.cwd().resolve())
dataset_incubator = workspace / 'dataset-incubator'

pop_path = dataset_incubator / 'out' / 'data' / 'mart' / 'popolazione_istat_comunale_2019_2025' / '2025' / 'popolazione_by_comune.parquet'
suolo_path = dataset_incubator / 'out' / 'data' / 'mart' / 'ispra_consumo_suolo' / '2024' / 'mart_comuni.parquet'

pop_path, suolo_path


In [ ]:
con = duckdb.connect()

query = '''
with pop as (
    select
        codice_comune,
        comune,
        popolazione_residente
    from read_parquet(?)
    where lower(comune) = 'abbiategrasso'
),
suolo as (
    select
        lpad(pro_com, 6, '0') as codice_comune,
        comune,
        stock_ha_2024,
        stock_pct_2024,
        incremento_ha_2023_2024
    from read_parquet(?)
    where lower(comune) = 'abbiategrasso'
)
select
    pop.codice_comune,
    pop.comune,
    popolazione_residente,
    stock_ha_2024,
    stock_pct_2024,
    incremento_ha_2023_2024,
    round(stock_ha_2024 * 10000.0 / popolazione_residente, 2) as mq_suolo_consumato_per_residente,
    round(incremento_ha_2023_2024 * 10000.0 / popolazione_residente, 2) as mq_incremento_per_residente
from pop
join suolo using (codice_comune)
'''

snapshot = con.execute(query, [pop_path.as_posix(), suolo_path.as_posix()]).df()
snapshot


## Lettura minima

Questa prima fotografia serve solo a fissare un punto di partenza locale:

- quanti residenti ha oggi Abbiategrasso
- quanta superficie consumata emerge dall'ultimo rilascio ISPRA
- quale segnale da' l'incremento piu' recente, anche rapportato ai residenti

Non e' ancora un confronto territoriale. Per il passo successivo conviene aggiungere almeno un benchmark con altri comuni vicini.


In [ ]:
snapshot.T
